# GALVAT Assistive29 Colab Training Notebook
This notebook trains the GALVAT baseline model on `galvat_openimages_v7` using Colab GPU and writes structured artifacts for downstream agent analysis.
It is designed for:
- dataset path: `/content/drive/MyDrive/galvat_openimages_v7`
- class schema: 29 classes (`assistive29`)
- baseline family: YOLO26s transfer learning


## 1) Runtime Setup
Run this section first. It mounts Google Drive and installs required dependencies.


In [ ]:
# Colab runtime dependencies
!pip -q install ultralytics==8.3.0 pyyaml==6.0.2 pandas==2.2.2 google

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Configuration
Edit paths/hyperparameters if needed before training.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import yaml
from ultralytics import YOLO

# -------- Core paths --------
DATASET_ROOT = Path('/content/drive/MyDrive/galvat_openimages_v7')
DATA_YAML = DATASET_ROOT / 'config' / 'data.yaml'
CLASS_TO_IDX_JSON = DATASET_ROOT / 'config' / 'class_to_idx.json'
CLASSES_TXT = DATASET_ROOT / 'config' / 'classes.txt'
CLASS_STATS_JSON = DATASET_ROOT / 'filtered' / 'stats' / 'class_stats.json'
SPLIT_STATS_JSON = DATASET_ROOT / 'filtered' / 'stats' / 'split_stats.json'
CLASS_SUPPORT_JSON = DATASET_ROOT / 'filtered' / 'stats' / 'class_support_report.json'

# Optional: mirror artifacts into your repo for agents to read directly
REPO_ROOT = Path('/content/drive/MyDrive/ml-agent-workbench')  # set to None to disable mirror

# -------- Run identity --------
RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = f'galvat_run_v2_baseline_yolo26s_transfer_roi_assistive29_colab_{RUN_TS}'
RUN_ROOT = DATASET_ROOT / 'experiments' / 'colab_runs' / RUN_ID

RUN_DIRS = {
    'root': RUN_ROOT,
    'logs': RUN_ROOT / 'logs',
    'metrics': RUN_ROOT / 'metrics',
    'models': RUN_ROOT / 'models',
    'exports': RUN_ROOT / 'exports',
    'env': RUN_ROOT / 'env',
    'config': RUN_ROOT / 'config',
    'reports': RUN_ROOT / 'reports',
    'snapshots': RUN_ROOT / 'snapshots',
}
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# -------- Baseline hyperparameters --------
TRAIN_CFG = {
    'model': 'yolo26s.pt',
    'imgsz': 640,
    'epochs': 100,
    'batch': 16,
    'workers': 8,
    'optimizer': 'Adam',
    'lr0': 0.001,
    'freeze': 10,
    'augment': False,
    'patience': 20,
    'seed': 1337,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'project': str(RUN_DIRS['models']),
    'name': 'train',
    'exist_ok': True,
}

print('RUN_ID:', RUN_ID)
print('RUN_ROOT:', RUN_ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3) Utility Helpers
These functions standardize artifact writing and hashing.


In [ ]:
def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def safe_read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def run_cmd(cmd: list[str]) -> dict:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return {
        'cmd': cmd,
        'returncode': proc.returncode,
        'stdout': proc.stdout[-20000:],
        'stderr': proc.stderr[-20000:],
    }


## 4) Dataset Validation + Snapshot
Validates expected dataset files and writes a reproducible snapshot for agent analysis.


In [ ]:
required_files = [
    DATA_YAML,
    CLASS_TO_IDX_JSON,
    CLASSES_TXT,
    CLASS_STATS_JSON,
    SPLIT_STATS_JSON,
    CLASS_SUPPORT_JSON,
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required dataset files:\n' + '\n'.join(missing))

class_to_idx = safe_read_json(CLASS_TO_IDX_JSON)
classes_txt = [x.strip() for x in CLASSES_TXT.read_text(encoding='utf-8').splitlines() if x.strip()]
class_stats = safe_read_json(CLASS_STATS_JSON)
split_stats = safe_read_json(SPLIT_STATS_JSON)
class_support = safe_read_json(CLASS_SUPPORT_JSON)
data_yaml = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))

if len(classes_txt) != 29:
    raise ValueError(f'Expected 29 classes, found {len(classes_txt)} in classes.txt')
if len(class_to_idx) != 29:
    raise ValueError(f'Expected 29 classes, found {len(class_to_idx)} in class_to_idx.json')

dataset_snapshot = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'run_id': RUN_ID,
    'dataset_root': str(DATASET_ROOT),
    'data_yaml': str(DATA_YAML),
    'class_count': len(classes_txt),
    'classes': classes_txt,
    'class_to_idx': class_to_idx,
    'split_stats': split_stats,
    'class_stats': class_stats,
    'class_support': class_support,
}
write_json(RUN_DIRS['snapshots'] / 'dataset_snapshot.json', dataset_snapshot)

print('Dataset snapshot written:', RUN_DIRS['snapshots'] / 'dataset_snapshot.json')
print('Train images:', split_stats['train']['num_images'])
print('Validation images:', split_stats['validation']['num_images'])
print('Test images:', split_stats['test']['num_images'])
print('Well-supported classes:', len(class_support.get('well_supported', [])))
print('Needs-more-images classes:', len(class_support.get('needs_more_images', [])))


## 5) Environment Capture
Captures software/hardware/runtime details for reproducibility.


In [ ]:
env_payload = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'python_version': sys.version,
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device_count': torch.cuda.device_count(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
write_json(RUN_DIRS['env'] / 'environment.json', env_payload)

# nvidia-smi capture (if GPU)
if torch.cuda.is_available():
    smi = run_cmd(['nvidia-smi'])
    write_json(RUN_DIRS['env'] / 'nvidia_smi.json', smi)

# pip freeze for exact package record
pip_freeze = run_cmd([sys.executable, '-m', 'pip', 'freeze'])
write_json(RUN_DIRS['env'] / 'pip_freeze.json', pip_freeze)

print('Environment artifacts written to', RUN_DIRS['env'])


## 6) Persist Run Config
Writes the exact training configuration used for this run.


In [ ]:
run_config_payload = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'dataset_root': str(DATASET_ROOT),
    'data_yaml': str(DATA_YAML),
    'train_config': TRAIN_CFG,
    'class_support_report': str(CLASS_SUPPORT_JSON),
}
write_json(RUN_DIRS['config'] / 'run_config.json', run_config_payload)
print('Saved run config:', RUN_DIRS['config'] / 'run_config.json')


## 7) Train
Runs YOLO26s training on Colab GPU. This can take a while.


In [ ]:
model = YOLO(TRAIN_CFG['model'])

train_args = dict(TRAIN_CFG)
train_args['data'] = str(DATA_YAML)

write_json(RUN_DIRS['config'] / 'train_args_effective.json', train_args)

train_results = model.train(**train_args)

train_run_dir = Path(train_results.save_dir)
summary = {
    'run_id': RUN_ID,
    'status': 'completed',
    'train_run_dir': str(train_run_dir),
    'best_weights': str(train_run_dir / 'weights' / 'best.pt'),
    'last_weights': str(train_run_dir / 'weights' / 'last.pt'),
    'results_csv': str(train_run_dir / 'results.csv'),
    'args_yaml': str(train_run_dir / 'args.yaml'),
}
write_json(RUN_DIRS['metrics'] / 'train_summary.json', summary)

print('Training complete. Train dir:', train_run_dir)


## 8) Validation (Val + Test)
Runs post-training evaluation and saves structured metrics.


In [ ]:
best_pt = Path(json.loads((RUN_DIRS['metrics'] / 'train_summary.json').read_text())['best_weights'])
if not best_pt.exists():
    raise FileNotFoundError(f'best.pt not found: {best_pt}')

best_model = YOLO(str(best_pt))

val_res = best_model.val(data=str(DATA_YAML), split='val', imgsz=TRAIN_CFG['imgsz'])
test_res = best_model.val(data=str(DATA_YAML), split='test', imgsz=TRAIN_CFG['imgsz'])

def extract_metrics(res, split_name: str):
    metrics = {
        'split': split_name,
        'precision': float(getattr(res.box, 'mp', float('nan'))),
        'recall': float(getattr(res.box, 'mr', float('nan'))),
        'map50': float(getattr(res.box, 'map50', float('nan'))),
        'map50_95': float(getattr(res.box, 'map', float('nan'))),
    }
    return metrics

val_metrics = extract_metrics(val_res, 'val')
test_metrics = extract_metrics(test_res, 'test')
write_json(RUN_DIRS['metrics'] / 'val_metrics_summary.json', val_metrics)
write_json(RUN_DIRS['metrics'] / 'test_metrics_summary.json', test_metrics)

# Per-class map values (if available)
per_class_payload = {}
if hasattr(val_res.box, 'maps') and val_res.box.maps is not None:
    maps = list(val_res.box.maps)
    names = [str(x) for x in yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))['names']]
    per_class_payload = {
        names[i]: float(maps[i]) for i in range(min(len(names), len(maps)))
    }
    write_json(RUN_DIRS['metrics'] / 'val_per_class_map50_95.json', per_class_payload)
    pd.DataFrame([{'class': k, 'map50_95': v} for k, v in per_class_payload.items()]).to_csv(
        RUN_DIRS['metrics'] / 'val_per_class_map50_95.csv', index=False
    )

print('Validation summaries written to', RUN_DIRS['metrics'])


## 9) ONNX Export + File Manifest
Exports ONNX and writes SHA256 manifests for all major artifacts.


In [ ]:
best_pt = Path(json.loads((RUN_DIRS['metrics'] / 'train_summary.json').read_text())['best_weights'])
best_model = YOLO(str(best_pt))
onnx_out = best_model.export(format='onnx', imgsz=TRAIN_CFG['imgsz'], opset=13, simplify=True)
onnx_path = Path(str(onnx_out))

export_summary = {
    'onnx_path': str(onnx_path),
    'onnx_exists': onnx_path.exists(),
}
write_json(RUN_DIRS['exports'] / 'onnx_export_summary.json', export_summary)

artifact_candidates = [
    best_pt,
    Path(json.loads((RUN_DIRS['metrics'] / 'train_summary.json').read_text())['last_weights']),
    onnx_path,
    Path(json.loads((RUN_DIRS['metrics'] / 'train_summary.json').read_text())['results_csv']),
    RUN_DIRS['metrics'] / 'train_summary.json',
    RUN_DIRS['metrics'] / 'val_metrics_summary.json',
    RUN_DIRS['metrics'] / 'test_metrics_summary.json',
    RUN_DIRS['config'] / 'run_config.json',
    RUN_DIRS['snapshots'] / 'dataset_snapshot.json',
    RUN_DIRS['env'] / 'environment.json',
]

manifest_rows = []
for p in artifact_candidates:
    row = {
        'path': str(p),
        'exists': p.exists(),
        'size_bytes': p.stat().st_size if p.exists() else None,
        'sha256': sha256_file(p) if p.exists() else None,
    }
    manifest_rows.append(row)

write_json(RUN_DIRS['reports'] / 'artifact_manifest.json', {'files': manifest_rows})
print('Artifact manifest written:', RUN_DIRS['reports'] / 'artifact_manifest.json')


## 10) Interpretation Report for Agents
Creates a compact machine-readable and human-readable summary that ties metrics to class support status.


In [ ]:
support = safe_read_json(CLASS_SUPPORT_JSON)
well_supported = set(support.get('well_supported', []))
needs_more = set(support.get('needs_more_images', []))

val_metrics = safe_read_json(RUN_DIRS['metrics'] / 'val_metrics_summary.json')
test_metrics = safe_read_json(RUN_DIRS['metrics'] / 'test_metrics_summary.json')
per_class_path = RUN_DIRS['metrics'] / 'val_per_class_map50_95.json'
per_class = safe_read_json(per_class_path) if per_class_path.exists() else {}

per_class_labeled = []
for cls, score in per_class.items():
    per_class_labeled.append({
        'class': cls,
        'val_map50_95': score,
        'support_bucket': 'well_supported' if cls in well_supported else ('needs_more_images' if cls in needs_more else 'unknown')
    })

analysis_payload = {
    'run_id': RUN_ID,
    'dataset_version': 'v7-assistive29-balanced-v2',
    'summary_metrics': {'val': val_metrics, 'test': test_metrics},
    'support_buckets': {
        'well_supported': sorted(well_supported),
        'needs_more_images': sorted(needs_more),
    },
    'per_class_val_map50_95_labeled': per_class_labeled,
    'interpretation_rules': [
        'Do not treat low performance in needs_more_images classes as architecture failure without data-support analysis.',
        'Prioritize stability and false-positive behavior on well_supported classes for baseline promotion decisions.'
    ]
}
write_json(RUN_DIRS['reports'] / 'agent_analysis_bundle.json', analysis_payload)

md_lines = [
    f"# GALVAT Colab Run Summary\n",
    f"- run_id: `{RUN_ID}`",
    f"- dataset_version: `v7-assistive29-balanced-v2`",
    f"- val precision/recall/map50/map50-95: {val_metrics['precision']:.4f} / {val_metrics['recall']:.4f} / {val_metrics['map50']:.4f} / {val_metrics['map50_95']:.4f}",
    f"- test precision/recall/map50/map50-95: {test_metrics['precision']:.4f} / {test_metrics['recall']:.4f} / {test_metrics['map50']:.4f} / {test_metrics['map50_95']:.4f}",
    f"- well_supported classes: {len(well_supported)}",
    f"- needs_more_images classes: {len(needs_more)}",
    "",
    "## Key Paths",
    f"- run_root: `{RUN_ROOT}`",
    f"- train_summary: `{RUN_DIRS['metrics'] / 'train_summary.json'}`",
    f"- metrics_bundle: `{RUN_DIRS['reports'] / 'agent_analysis_bundle.json'}`",
    f"- artifact_manifest: `{RUN_DIRS['reports'] / 'artifact_manifest.json'}`",
]
(RUN_DIRS['reports'] / 'run_summary.md').write_text('\n'.join(md_lines), encoding='utf-8')

print('Agent analysis bundle:', RUN_DIRS['reports'] / 'agent_analysis_bundle.json')
print('Run summary markdown:', RUN_DIRS['reports'] / 'run_summary.md')


## 11) Build Background-Only ROI Slice (Auto)
Auto-generates a background-only ROI slice from existing images by sampling RoIs that avoid all GT boxes in YOLO labels.


In [ ]:
# Auto-build background-only ROI slice from existing labels (standalone)
import os
import json
import random
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def _pick_existing_run_dir(dataset_root: Path) -> Path:
    runs_root = dataset_root / 'experiments' / 'colab_runs'
    if not runs_root.exists():
        raise FileNotFoundError(f'Run root not found: {runs_root}')
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No existing run folders found in: {runs_root}')
    # Prefer most recently modified run dir.
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


# Set GALVAT_DATASET_ROOT if your dataset lives somewhere else.
DATASET_ROOT = Path(os.environ.get('GALVAT_DATASET_ROOT', '/content/drive/MyDrive/galvat_openimages_v7'))

# Optional override: set GALVAT_EXISTING_RUN_DIR to pin a specific run folder.
existing_run_override = os.environ.get('GALVAT_EXISTING_RUN_DIR', '').strip()
if existing_run_override:
    RUN_ROOT = Path(existing_run_override)
else:
    RUN_ROOT = _pick_existing_run_dir(DATASET_ROOT)

if not RUN_ROOT.exists():
    raise FileNotFoundError(f'RUN_ROOT does not exist: {RUN_ROOT}')

RUN_ID = RUN_ROOT.name
RUN_DIRS = {
    'root': RUN_ROOT,
    'logs': RUN_ROOT / 'logs',
    'metrics': RUN_ROOT / 'metrics',
    'models': RUN_ROOT / 'models',
    'exports': RUN_ROOT / 'exports',
    'env': RUN_ROOT / 'env',
    'config': RUN_ROOT / 'config',
    'reports': RUN_ROOT / 'reports',
    'snapshots': RUN_ROOT / 'snapshots',
}
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Expose for next cell if executed sequentially.
globals()['DATASET_ROOT'] = DATASET_ROOT
globals()['RUN_ROOT'] = RUN_ROOT
globals()['RUN_ID'] = RUN_ID
globals()['RUN_DIRS'] = RUN_DIRS
globals()['write_json'] = _write_json

BACKGROUND_SLICE_PATH = DATASET_ROOT / 'manifests' / 'galvat_background_only_roi_slice_v1.json'
BACKGROUND_SLICE_PATH.parent.mkdir(parents=True, exist_ok=True)

SLICE_BUILD_CFG = {
    'seed': 1337,
    'target_rois_per_split': {
        'train': int(os.environ.get('GALVAT_BG_TARGET_TRAIN', '1000')),
        'validation': int(os.environ.get('GALVAT_BG_TARGET_VAL', '200')),
        'test': int(os.environ.get('GALVAT_BG_TARGET_TEST', '200')),
    },
    'max_rois_per_image': int(os.environ.get('GALVAT_BG_MAX_ROIS_PER_IMAGE', '2')),
    'max_attempts_per_roi': int(os.environ.get('GALVAT_BG_MAX_ATTEMPTS_PER_ROI', '80')),
    'min_roi_px': int(os.environ.get('GALVAT_BG_MIN_ROI_PX', '96')),
    'min_scale': float(os.environ.get('GALVAT_BG_MIN_SCALE', '0.12')),
    'max_scale': float(os.environ.get('GALVAT_BG_MAX_SCALE', '0.35')),
    'max_iou_with_gt': float(os.environ.get('GALVAT_BG_MAX_IOU_GT', '0.0')),
}

rng = random.Random(SLICE_BUILD_CFG['seed'])


def _xywhn_to_xyxy_abs(xc, yc, bw, bh, w, h):
    x1 = int(round((xc - bw / 2.0) * w))
    y1 = int(round((yc - bh / 2.0) * h))
    x2 = int(round((xc + bw / 2.0) * w))
    y2 = int(round((yc + bh / 2.0) * h))
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(x1 + 1, min(x2, w))
    y2 = max(y1 + 1, min(y2, h))
    return (x1, y1, x2, y2)


def _load_gt_boxes(label_path, w, h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            _, xc, yc, bw, bh = parts[:5]
            xc = float(xc)
            yc = float(yc)
            bw = float(bw)
            bh = float(bh)
        except ValueError:
            continue
        boxes.append(_xywhn_to_xyxy_abs(xc, yc, bw, bh, w, h))
    return boxes


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
    area_b = max(1, (bx2 - bx1) * (by2 - by1))
    union = area_a + area_b - inter
    return float(inter) / float(max(1, union))


def _sample_background_roi(w, h, gt_boxes):
    min_rw = max(SLICE_BUILD_CFG['min_roi_px'], int(w * SLICE_BUILD_CFG['min_scale']))
    min_rh = max(SLICE_BUILD_CFG['min_roi_px'], int(h * SLICE_BUILD_CFG['min_scale']))
    max_rw = max(min_rw, int(w * SLICE_BUILD_CFG['max_scale']))
    max_rh = max(min_rh, int(h * SLICE_BUILD_CFG['max_scale']))

    max_rw = min(max_rw, w - 1)
    max_rh = min(max_rh, h - 1)

    if min_rw >= w or min_rh >= h:
        return None

    for _ in range(SLICE_BUILD_CFG['max_attempts_per_roi']):
        rw = rng.randint(min_rw, max_rw)
        rh = rng.randint(min_rh, max_rh)
        if rw >= w or rh >= h:
            continue

        x1 = rng.randint(0, w - rw)
        y1 = rng.randint(0, h - rh)
        roi = (x1, y1, x1 + rw, y1 + rh)

        max_iou = 0.0
        for gt in gt_boxes:
            max_iou = max(max_iou, _iou(roi, gt))
            if max_iou > SLICE_BUILD_CFG['max_iou_with_gt']:
                break

        if max_iou <= SLICE_BUILD_CFG['max_iou_with_gt']:
            return roi

    return None


def _list_images(split_dir):
    out = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        out.extend(split_dir.glob(ext))
    return sorted(set(out))


def _build_split(split_name):
    img_dir = DATASET_ROOT / 'images' / split_name
    label_dir = DATASET_ROOT / 'labels' / split_name

    if not img_dir.exists():
        raise FileNotFoundError(f'Missing images directory for split={split_name}: {img_dir}')

    imgs = _list_images(img_dir)
    rng.shuffle(imgs)

    target = int(SLICE_BUILD_CFG['target_rois_per_split'][split_name])
    entries = []
    attempted_images = 0

    for img_path in imgs:
        if len(entries) >= target:
            break

        attempted_images += 1

        with Image.open(img_path) as im:
            w, h = im.size

        label_path = label_dir / f'{img_path.stem}.txt'
        gt_boxes = _load_gt_boxes(label_path, w, h)

        for _ in range(SLICE_BUILD_CFG['max_rois_per_image']):
            if len(entries) >= target:
                break
            roi = _sample_background_roi(w, h, gt_boxes)
            if roi is None:
                break

            x1, y1, x2, y2 = roi
            entries.append({
                'image_id': img_path.stem,
                'image_path': f'images/{split_name}/{img_path.name}',
                'x1': int(x1),
                'y1': int(y1),
                'x2': int(x2),
                'y2': int(y2),
                'source': 'auto_background_sampler_v1',
                'gt_boxes_considered': int(len(gt_boxes)),
            })

    return entries, {'attempted_images': attempted_images, 'available_images': len(imgs), 'target_rois': target}


splits = {}
build_stats = {}
for split_name in ['train', 'validation', 'test']:
    entries, stats = _build_split(split_name)
    splits[split_name] = entries
    build_stats[split_name] = {
        **stats,
        'generated_rois': len(entries),
        'hit_target': len(entries) >= stats['target_rois'],
    }

slice_payload = {
    'slice_id': 'galvat_background_only_roi_slice_v1',
    'dataset_id': 'galvat_dataset_manifest_v1',
    'domain': 'galvat',
    'version': 'v7-assistive29-balanced-v2',
    'status': 'auto_generated_background_rois_ready_for_fp_eval',
    'splits': splits,
    'build_config': SLICE_BUILD_CFG,
    'build_stats': build_stats,
    'created_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'note': 'Auto-sampled RoIs with IoU=0.0 against labeled GT boxes; manual review recommended for a subset before deployment claims.',
}

_write_json(BACKGROUND_SLICE_PATH, slice_payload)
_write_json(RUN_DIRS['metrics'] / 'background_slice_build_summary.json', {
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'slice_path': str(BACKGROUND_SLICE_PATH),
    'status': slice_payload['status'],
    'build_stats': build_stats,
})

print('Using existing run:', RUN_ROOT)
print('Background slice written:', BACKGROUND_SLICE_PATH)
for split_name in ['train', 'validation', 'test']:
    s = build_stats[split_name]
    print(f"{split_name}: generated={s['generated_rois']} target={s['target_rois']} hit_target={s['hit_target']}")
print('Build summary:', RUN_DIRS['metrics'] / 'background_slice_build_summary.json')




## 12) Background-Only False Positive Benchmark
Evaluates false-positive behavior on background-only RoIs/images and writes structured outputs for agents.


In [ ]:
# Background-only false-positive benchmark on existing trained model (standalone)
import os
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path


# Install missing deps if this is a fresh runtime where earlier setup cells were not run.
def _ensure_pkg(import_name: str, pip_name: str | None = None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name or import_name
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)


_ensure_pkg('ultralytics', 'ultralytics==8.3.0')
_ensure_pkg('pandas', 'pandas==2.2.2')
_ensure_pkg('PIL', 'Pillow')

import pandas as pd
from PIL import Image
from ultralytics import YOLO


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def _safe_read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def _pick_existing_run_dir(dataset_root: Path) -> Path:
    runs_root = dataset_root / 'experiments' / 'colab_runs'
    if not runs_root.exists():
        raise FileNotFoundError(f'Run root not found: {runs_root}')
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No existing run folders found in: {runs_root}')
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


if 'DATASET_ROOT' not in globals() or 'RUN_ROOT' not in globals() or 'RUN_DIRS' not in globals():
    DATASET_ROOT = Path(os.environ.get('GALVAT_DATASET_ROOT', '/content/drive/MyDrive/galvat_openimages_v7'))
    existing_run_override = os.environ.get('GALVAT_EXISTING_RUN_DIR', '').strip()
    RUN_ROOT = Path(existing_run_override) if existing_run_override else _pick_existing_run_dir(DATASET_ROOT)
    RUN_ID = RUN_ROOT.name
    RUN_DIRS = {
        'root': RUN_ROOT,
        'logs': RUN_ROOT / 'logs',
        'metrics': RUN_ROOT / 'metrics',
        'models': RUN_ROOT / 'models',
        'exports': RUN_ROOT / 'exports',
        'env': RUN_ROOT / 'env',
        'config': RUN_ROOT / 'config',
        'reports': RUN_ROOT / 'reports',
        'snapshots': RUN_ROOT / 'snapshots',
    }
    for p in RUN_DIRS.values():
        p.mkdir(parents=True, exist_ok=True)

# Resolve best checkpoint from existing run
train_summary_path = RUN_DIRS['metrics'] / 'train_summary.json'
if train_summary_path.exists():
    best_pt = Path(_safe_read_json(train_summary_path).get('best_weights', ''))
else:
    best_pt = RUN_ROOT / 'models' / 'train' / 'weights' / 'best.pt'

if not best_pt.exists():
    raise FileNotFoundError(
        f'Could not find best.pt for existing run. Looked at: {best_pt}. '
        'Set GALVAT_EXISTING_RUN_DIR to the correct run folder.'
    )

# Try to read imgsz from existing train args; fallback to 640.
imgsz = 640
train_args_path = RUN_DIRS['config'] / 'train_args_effective.json'
if train_args_path.exists():
    try:
        imgsz = int(_safe_read_json(train_args_path).get('imgsz', 640))
    except Exception:
        imgsz = 640

BACKGROUND_SLICE_CANDIDATES = [
    DATASET_ROOT / 'manifests' / 'galvat_background_only_roi_slice_v1.json',
    DATASET_ROOT / 'filtered' / 'stats' / 'galvat_background_only_roi_slice_v1.json',
]

background_slice_path = None
for cand in BACKGROUND_SLICE_CANDIDATES:
    if cand.exists():
        background_slice_path = cand
        break

if background_slice_path is None:
    raise FileNotFoundError(
        'Could not find background-only slice JSON. Expected one of:\n' +
        '\n'.join(str(p) for p in BACKGROUND_SLICE_CANDIDATES)
    )

background_slice = _safe_read_json(background_slice_path)
splits_payload = background_slice.get('splits', {})
if not isinstance(splits_payload, dict):
    raise ValueError('Background slice JSON must contain an object at key "splits".')

FP_CFG = {
    'conf': float(os.environ.get('GALVAT_FP_CONF', '0.25')),
    'iou': float(os.environ.get('GALVAT_FP_IOU', '0.45')),
    'imgsz': imgsz,
    'max_det': int(os.environ.get('GALVAT_FP_MAX_DET', '100')),
    'high_conf_threshold': float(os.environ.get('GALVAT_FP_HIGH_CONF', '0.70')),
}

max_total_eval = int(os.environ.get('GALVAT_FP_MAX_TOTAL', '0'))  # 0 means no cap

SPLIT_ALIAS = {'val': 'validation', 'validation': 'validation', 'test': 'test', 'train': 'train'}

def _resolve_image_path(entry, split_name):
    split_norm = SPLIT_ALIAS.get(str(split_name).lower(), str(split_name).lower())
    split_dir = DATASET_ROOT / 'images' / split_norm

    if isinstance(entry, str):
        s = entry.strip()
        if not s:
            return None
        p = Path(s)
        if p.exists():
            return p
        if s.startswith('images/'):
            p2 = DATASET_ROOT / s
            if p2.exists():
                return p2
        if '/' in s:
            p3 = DATASET_ROOT / s
            if p3.exists():
                return p3
        if p.suffix:
            p4 = split_dir / s
            if p4.exists():
                return p4
        p5 = split_dir / f'{s}.jpg'
        if p5.exists():
            return p5
        return None

    if isinstance(entry, dict):
        for key in ['image_path', 'path', 'file_path', 'filepath']:
            raw = entry.get(key)
            if isinstance(raw, str) and raw.strip():
                p = Path(raw)
                if p.exists():
                    return p
                p2 = DATASET_ROOT / raw
                if p2.exists():
                    return p2
        image_id = entry.get('image_id')
        if isinstance(image_id, str) and image_id.strip():
            p = split_dir / f'{image_id}.jpg'
            if p.exists():
                return p
    return None


def _resolve_roi(entry, img_w, img_h):
    if not isinstance(entry, dict):
        return (0, 0, img_w, img_h)

    if all(k in entry for k in ['x1', 'y1', 'x2', 'y2']):
        x1, y1, x2, y2 = [int(round(float(entry[k]))) for k in ['x1', 'y1', 'x2', 'y2']]
    elif all(k in entry for k in ['x_center', 'y_center', 'width', 'height']):
        xc = float(entry['x_center']) * img_w
        yc = float(entry['y_center']) * img_h
        bw = float(entry['width']) * img_w
        bh = float(entry['height']) * img_h
        x1 = int(round(xc - bw / 2.0))
        y1 = int(round(yc - bh / 2.0))
        x2 = int(round(xc + bw / 2.0))
        y2 = int(round(yc + bh / 2.0))
    else:
        return (0, 0, img_w, img_h)

    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(x1 + 1, min(x2, img_w))
    y2 = max(y1 + 1, min(y2, img_h))
    return (x1, y1, x2, y2)


fp_model = YOLO(str(best_pt))

rows = []
class_counter = Counter()
unresolved_entries = []

for split_name, entries in splits_payload.items():
    if not isinstance(entries, list):
        continue
    for idx, entry in enumerate(entries):
        if max_total_eval > 0 and len(rows) >= max_total_eval:
            break

        image_path = _resolve_image_path(entry, split_name)
        if image_path is None or not image_path.exists():
            unresolved_entries.append({'split': split_name, 'index': idx, 'entry': entry})
            continue

        with Image.open(image_path) as im:
            im = im.convert('RGB')
            w, h = im.size
            x1, y1, x2, y2 = _resolve_roi(entry, w, h)
            crop = im.crop((x1, y1, x2, y2))

        pred = fp_model.predict(
            source=crop,
            conf=FP_CFG['conf'],
            iou=FP_CFG['iou'],
            imgsz=FP_CFG['imgsz'],
            max_det=FP_CFG['max_det'],
            verbose=False,
        )[0]

        names = pred.names if hasattr(pred, 'names') else {}
        det_count = int(len(pred.boxes)) if getattr(pred, 'boxes', None) is not None else 0
        high_conf = 0

        if det_count > 0:
            cls_ids = pred.boxes.cls.tolist()
            confs = pred.boxes.conf.tolist()
            for cls_id, conf in zip(cls_ids, confs):
                cls_name = names.get(int(cls_id), str(int(cls_id)))
                class_counter[str(cls_name)] += 1
                if float(conf) >= FP_CFG['high_conf_threshold']:
                    high_conf += 1

        rows.append({
            'split': split_name,
            'image_path': str(image_path),
            'roi_x1': x1,
            'roi_y1': y1,
            'roi_x2': x2,
            'roi_y2': y2,
            'detections': det_count,
            'has_detection': int(det_count > 0),
            'high_conf_detections': high_conf,
        })
    if max_total_eval > 0 and len(rows) >= max_total_eval:
        break

if not rows:
    raise ValueError(
        'No evaluable background-only entries found. '
        'Run the previous cell first to populate manifests/galvat_background_only_roi_slice_v1.json.'
    )

fp_df = pd.DataFrame(rows)
summary = {
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'best_weights': str(best_pt),
    'background_slice_path': str(background_slice_path),
    'eval_config': FP_CFG,
    'num_rois_evaluated': int(len(fp_df)),
    'num_rois_with_any_detection': int(fp_df['has_detection'].sum()),
    'pct_rois_with_any_detection': float(fp_df['has_detection'].mean()),
    'total_false_positive_detections': int(fp_df['detections'].sum()),
    'mean_fp_detections_per_roi': float(fp_df['detections'].mean()),
    'median_fp_detections_per_roi': float(fp_df['detections'].median()),
    'max_fp_detections_in_single_roi': int(fp_df['detections'].max()),
    'high_conf_fp_detections_total': int(fp_df['high_conf_detections'].sum()),
    'top_fp_classes': [
        {'class': cls, 'count': int(cnt)} for cls, cnt in class_counter.most_common(15)
    ],
    'by_split': {},
    'unresolved_entries_count': int(len(unresolved_entries)),
}

for split_name, part in fp_df.groupby('split'):
    summary['by_split'][split_name] = {
        'num_rois': int(len(part)),
        'num_rois_with_any_detection': int(part['has_detection'].sum()),
        'pct_rois_with_any_detection': float(part['has_detection'].mean()),
        'mean_fp_detections_per_roi': float(part['detections'].mean()),
        'high_conf_fp_detections_total': int(part['high_conf_detections'].sum()),
    }

_write_json(RUN_DIRS['metrics'] / 'background_fp_summary.json', summary)
fp_df.to_csv(RUN_DIRS['metrics'] / 'background_fp_details.csv', index=False)
_write_json(RUN_DIRS['metrics'] / 'background_fp_unresolved_entries.json', {'entries': unresolved_entries})

print('Using existing run:', RUN_ROOT)
print('Background FP summary:', RUN_DIRS['metrics'] / 'background_fp_summary.json')
print('Background FP details:', RUN_DIRS['metrics'] / 'background_fp_details.csv')
print('Evaluated ROIs:', len(fp_df))
print('ROIs with any detection:', int(fp_df['has_detection'].sum()))
print('Mean FP detections/ROI:', round(float(fp_df['detections'].mean()), 4))
print('High-confidence FP detections total:', int(fp_df['high_conf_detections'].sum()))




## 13) Split Integrity Audit (Issue 2)
Audits split leakage by image ID overlap, optional content-hash overlap, and availability of auditable scene/source metadata.


In [ ]:
# Split integrity audit (standalone, no retraining)
import os
import json
import hashlib
from itertools import combinations
from pathlib import Path


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def _pick_existing_run_dir(dataset_root: Path) -> Path:
    runs_root = dataset_root / 'experiments' / 'colab_runs'
    if not runs_root.exists():
        raise FileNotFoundError(f'Run root not found: {runs_root}')
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No existing run folders found in: {runs_root}')
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def _list_image_files(split_dir: Path):
    files = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        files.extend(split_dir.glob(ext))
    return sorted(set(files))


def _read_ids_txt(path: Path):
    if not path.exists():
        return []
    return [ln.strip() for ln in path.read_text(encoding='utf-8', errors='ignore').splitlines() if ln.strip()]


def _sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _pair_overlap(a: set, b: set):
    inter = a.intersection(b)
    return {
        'count': len(inter),
        'sample': sorted(list(inter))[:20],
    }


DATASET_ROOT = Path(os.environ.get('GALVAT_DATASET_ROOT', '/content/drive/MyDrive/galvat_openimages_v7'))
existing_run_override = os.environ.get('GALVAT_EXISTING_RUN_DIR', '').strip()
RUN_ROOT = Path(existing_run_override) if existing_run_override else _pick_existing_run_dir(DATASET_ROOT)
RUN_ID = RUN_ROOT.name

RUN_DIRS = {
    'root': RUN_ROOT,
    'logs': RUN_ROOT / 'logs',
    'metrics': RUN_ROOT / 'metrics',
    'models': RUN_ROOT / 'models',
    'exports': RUN_ROOT / 'exports',
    'env': RUN_ROOT / 'env',
    'config': RUN_ROOT / 'config',
    'reports': RUN_ROOT / 'reports',
    'snapshots': RUN_ROOT / 'snapshots',
}
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

split_dirs = {
    'train': DATASET_ROOT / 'images' / 'train',
    'validation': DATASET_ROOT / 'images' / 'validation',
    'test': DATASET_ROOT / 'images' / 'test',
}

split_files = {}
split_ids = {}
missing_split_dirs = []
for split, d in split_dirs.items():
    if not d.exists():
        missing_split_dirs.append(str(d))
        split_files[split] = []
        split_ids[split] = set()
        continue
    files = _list_image_files(d)
    split_files[split] = files
    split_ids[split] = {p.stem for p in files}

pairwise_id_overlap = {}
for a, b in combinations(['train', 'validation', 'test'], 2):
    pairwise_id_overlap[f'{a}__{b}'] = _pair_overlap(split_ids[a], split_ids[b])

triple_overlap = split_ids['train'].intersection(split_ids['validation']).intersection(split_ids['test'])

# Compare image IDs against filtered/image_ids/*.txt if available.
ids_txt_paths = {
    'train': DATASET_ROOT / 'filtered' / 'image_ids' / 'train_image_ids.txt',
    'validation': DATASET_ROOT / 'filtered' / 'image_ids' / 'validation_image_ids.txt',
    'test': DATASET_ROOT / 'filtered' / 'image_ids' / 'test_image_ids.txt',
}
ids_txt_compare = {}
for split in ['train', 'validation', 'test']:
    txt_ids = set(_read_ids_txt(ids_txt_paths[split]))
    dir_ids = split_ids[split]
    if not txt_ids:
        ids_txt_compare[split] = {
            'ids_txt_found': False,
            'ids_txt_path': str(ids_txt_paths[split]),
        }
        continue
    ids_txt_compare[split] = {
        'ids_txt_found': True,
        'ids_txt_path': str(ids_txt_paths[split]),
        'ids_txt_count': len(txt_ids),
        'image_dir_count': len(dir_ids),
        'missing_in_image_dir_count': len(txt_ids - dir_ids),
        'extra_in_image_dir_count': len(dir_ids - txt_ids),
        'missing_in_image_dir_sample': sorted(list(txt_ids - dir_ids))[:20],
        'extra_in_image_dir_sample': sorted(list(dir_ids - txt_ids))[:20],
    }

# Scene/source metadata auditability check.
scene_source_candidates = [
    DATASET_ROOT / 'manifests' / 'galvat_scene_split_registry_v1.json',
    DATASET_ROOT / 'filtered' / 'stats' / 'scene_source_map.csv',
    DATASET_ROOT / 'filtered' / 'stats' / 'scene_id_map.csv',
    DATASET_ROOT / 'filtered' / 'stats' / 'source_id_map.csv',
]
scene_source_found = [str(p) for p in scene_source_candidates if p.exists()]

scene_metadata_auditable = False
scene_metadata_reason = 'No explicit per-image scene/source table with stable IDs was found.'
if scene_source_found:
    scene_metadata_reason = (
        'Found candidate metadata files, but auditable leakage verification requires per-image scene/source IDs '
        'that can be joined across all splits.'
    )

# Optional expensive exact-content overlap audit across splits
run_content_hash_audit = os.environ.get('GALVAT_SPLIT_AUDIT_CONTENT_HASH', '0') == '1'
content_hash_summary = {
    'enabled': run_content_hash_audit,
    'pairwise_overlap_count': {},
    'pairwise_overlap_sample': {},
}

if run_content_hash_audit:
    hash_to_splits = {}
    hash_to_files = {}
    for split, files in split_files.items():
        for p in files:
            h = _sha256_file(p)
            hash_to_splits.setdefault(h, set()).add(split)
            hash_to_files.setdefault(h, []).append(str(p))

    pair_counts = {'train__validation': 0, 'train__test': 0, 'validation__test': 0}
    pair_samples = {'train__validation': [], 'train__test': [], 'validation__test': []}
    for h, splits in hash_to_splits.items():
        if len(splits) < 2:
            continue
        if 'train' in splits and 'validation' in splits:
            pair_counts['train__validation'] += 1
            if len(pair_samples['train__validation']) < 20:
                pair_samples['train__validation'].append({'sha256': h, 'files': hash_to_files[h][:4]})
        if 'train' in splits and 'test' in splits:
            pair_counts['train__test'] += 1
            if len(pair_samples['train__test']) < 20:
                pair_samples['train__test'].append({'sha256': h, 'files': hash_to_files[h][:4]})
        if 'validation' in splits and 'test' in splits:
            pair_counts['validation__test'] += 1
            if len(pair_samples['validation__test']) < 20:
                pair_samples['validation__test'].append({'sha256': h, 'files': hash_to_files[h][:4]})

    content_hash_summary['pairwise_overlap_count'] = pair_counts
    content_hash_summary['pairwise_overlap_sample'] = pair_samples

id_overlap_total = sum(v['count'] for v in pairwise_id_overlap.values()) + len(triple_overlap)
id_overlap_pass = id_overlap_total == 0

overall_status = 'pass'
if missing_split_dirs:
    overall_status = 'fail'
elif not id_overlap_pass:
    overall_status = 'fail'
elif not scene_metadata_auditable:
    overall_status = 'partial_pass'

report = {
    'audit_id': 'split_integrity_audit_v1',
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'dataset_root': str(DATASET_ROOT),
    'overall_status': overall_status,
    'checks': {
        'split_directories_present': {
            'pass': len(missing_split_dirs) == 0,
            'missing_dirs': missing_split_dirs,
        },
        'image_id_overlap': {
            'pass': id_overlap_pass,
            'pairwise': pairwise_id_overlap,
            'triple_overlap_count': len(triple_overlap),
            'triple_overlap_sample': sorted(list(triple_overlap))[:20],
        },
        'ids_txt_alignment': ids_txt_compare,
        'scene_source_metadata_auditable': {
            'pass': scene_metadata_auditable,
            'reason': scene_metadata_reason,
            'candidate_files_found': scene_source_found,
        },
        'content_hash_overlap': content_hash_summary,
    },
    'split_counts': {
        split: {
            'image_file_count': len(split_files[split]),
            'unique_image_id_count': len(split_ids[split]),
        }
        for split in ['train', 'validation', 'test']
    },
    'next_actions': [
        'Keep image ID overlap at zero across train/validation/test.',
        'Attach auditable per-image scene/source metadata (scene_id/source_id) to verify leakage risk explicitly.',
        'Optionally rerun this audit with GALVAT_SPLIT_AUDIT_CONTENT_HASH=1 for exact byte-level duplicate checks across splits.'
    ]
}

_write_json(RUN_DIRS['metrics'] / 'split_integrity_audit_v1.json', report)

md_lines = [
    '# Split Integrity Audit v1',
    f"- run_id: `{RUN_ID}`",
    f"- overall_status: `{overall_status}`",
    '',
    '## Summary',
    f"- split dirs present: `{len(missing_split_dirs) == 0}`",
    f"- image-id overlap pass: `{id_overlap_pass}`",
    f"- scene/source metadata auditable: `{scene_metadata_auditable}`",
    f"- content hash audit enabled: `{run_content_hash_audit}`",
    '',
    '## Pairwise image-id overlaps',
]
for key, v in pairwise_id_overlap.items():
    md_lines.append(f"- {key}: `{v['count']}`")
md_lines.append(f"- train__validation__test: `{len(triple_overlap)}`")

(RUN_DIRS['reports'] / 'split_integrity_audit_v1.md').write_text('\n'.join(md_lines), encoding='utf-8')

print('Split integrity audit JSON:', RUN_DIRS['metrics'] / 'split_integrity_audit_v1.json')
print('Split integrity audit Markdown:', RUN_DIRS['reports'] / 'split_integrity_audit_v1.md')
print('Overall status:', overall_status)
print('Image-ID overlap pass:', id_overlap_pass)
print('Scene/source metadata auditable:', scene_metadata_auditable)



## 14) ROI Boundary-Adherence Evaluation (Issue 3)
Evaluates whether detections stay within RoI boundaries using a structured protocol over sampled object-centered RoIs from validation/test labels.


In [ ]:
# ROI boundary-adherence evaluation (Issue 3, standalone, no retraining)
import os
import json
import random
import subprocess
import sys
from collections import defaultdict
from pathlib import Path


def _ensure_pkg(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name], check=True)


_ensure_pkg('ultralytics', 'ultralytics==8.3.0')
_ensure_pkg('pandas', 'pandas==2.2.2')
_ensure_pkg('PIL', 'Pillow')

import pandas as pd
from PIL import Image
from ultralytics import YOLO


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def _safe_read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def _pick_existing_run_dir(dataset_root: Path) -> Path:
    runs_root = dataset_root / 'experiments' / 'colab_runs'
    if not runs_root.exists():
        raise FileNotFoundError(f'Run root not found: {runs_root}')
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No existing run folders found in: {runs_root}')
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def _resolve_best_weights(run_dirs: dict, run_root: Path) -> Path:
    env_model = os.environ.get('GALVAT_MODEL_PATH', '').strip()
    if env_model:
        p = Path(env_model)
        if p.exists():
            return p

    train_summary_path = run_dirs['metrics'] / 'train_summary.json'
    if train_summary_path.exists():
        try:
            p = Path(_safe_read_json(train_summary_path).get('best_weights', ''))
            if p.exists():
                return p
        except Exception:
            pass

    candidates = sorted(run_root.rglob('best.pt'), key=lambda x: x.stat().st_mtime, reverse=True)
    if candidates:
        return candidates[0]

    raise FileNotFoundError('Could not resolve best.pt. Set GALVAT_MODEL_PATH to your trained checkpoint.')


def _find_image_for_id(image_dir: Path, image_id: str):
    for ext in ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'):
        p = image_dir / f'{image_id}{ext}'
        if p.exists():
            return p
    return None


def _xywhn_to_xyxy_abs(xc, yc, bw, bh, w, h):
    x1 = int(round((xc - bw / 2.0) * w))
    y1 = int(round((yc - bh / 2.0) * h))
    x2 = int(round((xc + bw / 2.0) * w))
    y2 = int(round((yc + bh / 2.0) * h))
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(x1 + 1, min(x2, w))
    y2 = max(y1 + 1, min(y2, h))
    return x1, y1, x2, y2


def _expand_box(x1, y1, x2, y2, w, h, expand_ratio):
    bw = x2 - x1
    bh = y2 - y1
    ex = int(round(bw * expand_ratio))
    ey = int(round(bh * expand_ratio))
    nx1 = max(0, x1 - ex)
    ny1 = max(0, y1 - ey)
    nx2 = min(w, x2 + ex)
    ny2 = min(h, y2 + ey)
    nx2 = max(nx1 + 1, nx2)
    ny2 = max(ny1 + 1, ny2)
    return nx1, ny1, nx2, ny2


def _parse_label_file(label_path: Path, img_w: int, img_h: int):
    rows = []
    for line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls_id = int(float(parts[0]))
            xc = float(parts[1])
            yc = float(parts[2])
            bw = float(parts[3])
            bh = float(parts[4])
        except Exception:
            continue
        x1, y1, x2, y2 = _xywhn_to_xyxy_abs(xc, yc, bw, bh, img_w, img_h)
        rows.append({'cls_id': cls_id, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
    return rows


def _bbox_intersects(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return (max(ax1, bx1) < min(ax2, bx2)) and (max(ay1, by1) < min(ay2, by2))


DATASET_ROOT = Path(os.environ.get('GALVAT_DATASET_ROOT', '/content/drive/MyDrive/galvat_openimages_v7'))
existing_run_override = os.environ.get('GALVAT_EXISTING_RUN_DIR', '').strip()
RUN_ROOT = Path(existing_run_override) if existing_run_override else _pick_existing_run_dir(DATASET_ROOT)
RUN_ID = RUN_ROOT.name
RUN_DIRS = {
    'root': RUN_ROOT,
    'logs': RUN_ROOT / 'logs',
    'metrics': RUN_ROOT / 'metrics',
    'models': RUN_ROOT / 'models',
    'exports': RUN_ROOT / 'exports',
    'env': RUN_ROOT / 'env',
    'config': RUN_ROOT / 'config',
    'reports': RUN_ROOT / 'reports',
    'snapshots': RUN_ROOT / 'snapshots',
}
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

weights_path = _resolve_best_weights(RUN_DIRS, RUN_ROOT)
model = YOLO(str(weights_path))

cfg = {
    'seed': int(os.environ.get('GALVAT_ROI_BOUNDARY_SEED', '1337')),
    'splits': [x.strip() for x in os.environ.get('GALVAT_ROI_BOUNDARY_SPLITS', 'validation,test').split(',') if x.strip()],
    'target_rois_per_split': int(os.environ.get('GALVAT_ROI_BOUNDARY_TARGET_PER_SPLIT', '250')),
    'max_rois_per_image': int(os.environ.get('GALVAT_ROI_BOUNDARY_MAX_ROIS_PER_IMAGE', '2')),
    'roi_expand_ratio': float(os.environ.get('GALVAT_ROI_BOUNDARY_EXPAND', '0.15')),
    'boundary_margin_px': int(os.environ.get('GALVAT_ROI_BOUNDARY_MARGIN_PX', '8')),
    'conf': float(os.environ.get('GALVAT_ROI_BOUNDARY_CONF', '0.25')),
    'iou': float(os.environ.get('GALVAT_ROI_BOUNDARY_IOU', '0.45')),
    'imgsz': int(os.environ.get('GALVAT_ROI_BOUNDARY_IMGSZ', '640')),
    'max_det': int(os.environ.get('GALVAT_ROI_BOUNDARY_MAX_DET', '100')),
}

rng = random.Random(cfg['seed'])

roi_entries = []
for split in cfg['splits']:
    image_dir = DATASET_ROOT / 'images' / split
    label_dir = DATASET_ROOT / 'labels' / split
    if not image_dir.exists() or not label_dir.exists():
        continue

    label_files = list(label_dir.glob('*.txt'))
    rng.shuffle(label_files)
    split_count = 0

    for label_path in label_files:
        if split_count >= cfg['target_rois_per_split']:
            break

        image_id = label_path.stem
        image_path = _find_image_for_id(image_dir, image_id)
        if image_path is None:
            continue

        with Image.open(image_path) as im:
            w, h = im.size

        boxes = _parse_label_file(label_path, w, h)
        if not boxes:
            continue

        indices = list(range(len(boxes)))
        rng.shuffle(indices)
        for idx in indices[:max(1, cfg['max_rois_per_image'])]:
            if split_count >= cfg['target_rois_per_split']:
                break
            b = boxes[idx]
            x1, y1, x2, y2 = _expand_box(b['x1'], b['y1'], b['x2'], b['y2'], w, h, cfg['roi_expand_ratio'])
            roi_entries.append({
                'split': split,
                'image_id': image_id,
                'image_path': str(image_path),
                'gt_class_id': int(b['cls_id']),
                'roi_x1': int(x1),
                'roi_y1': int(y1),
                'roi_x2': int(x2),
                'roi_y2': int(y2),
            })
            split_count += 1

if not roi_entries:
    raise ValueError('No ROI entries created. Check labels/images paths and split names.')

rois_by_image = defaultdict(list)
for i, row in enumerate(roi_entries):
    rois_by_image[row['image_path']].append(i)

pred_cache = {}
for image_path in rois_by_image.keys():
    pred = model.predict(
        source=image_path,
        conf=cfg['conf'],
        iou=cfg['iou'],
        imgsz=cfg['imgsz'],
        max_det=cfg['max_det'],
        verbose=False,
    )[0]

    names = pred.names if hasattr(pred, 'names') else {}
    dets = []
    if getattr(pred, 'boxes', None) is not None and len(pred.boxes) > 0:
        xyxy = pred.boxes.xyxy.tolist()
        confs = pred.boxes.conf.tolist()
        classes = pred.boxes.cls.tolist()
        for box, conf, cls_id in zip(xyxy, confs, classes):
            x1, y1, x2, y2 = [float(v) for v in box]
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0
            cls_i = int(cls_id)
            dets.append({
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
                'cx': cx,
                'cy': cy,
                'conf': float(conf),
                'class_id': cls_i,
                'class_name': str(names.get(cls_i, cls_i)),
            })

    pred_cache[image_path] = dets

rows = []
for row in roi_entries:
    x1, y1, x2, y2 = row['roi_x1'], row['roi_y1'], row['roi_x2'], row['roi_y2']
    roi_box = (x1, y1, x2, y2)
    dets = pred_cache[row['image_path']]

    inside_center = 0
    outside_center = 0
    crossing_bbox = 0
    near_edge_inside = 0
    inside_classes = []

    for d in dets:
        inside = (x1 <= d['cx'] <= x2) and (y1 <= d['cy'] <= y2)
        intersects = _bbox_intersects(roi_box, (d['x1'], d['y1'], d['x2'], d['y2']))

        if inside:
            inside_center += 1
            inside_classes.append(d['class_name'])
            edge_dist = min(d['cx'] - x1, x2 - d['cx'], d['cy'] - y1, y2 - d['cy'])
            if edge_dist <= cfg['boundary_margin_px']:
                near_edge_inside += 1
        else:
            outside_center += 1

        if intersects and (not inside):
            crossing_bbox += 1

    total = inside_center + outside_center
    rows.append({
        **row,
        'fullframe_detection_count': int(total),
        'inside_center_count': int(inside_center),
        'outside_center_count': int(outside_center),
        'crossing_bbox_count': int(crossing_bbox),
        'near_edge_inside_count': int(near_edge_inside),
        'outside_share_before_filter': float(outside_center / total) if total > 0 else 0.0,
        'inside_classes_json': json.dumps(inside_classes),
    })

boundary_df = pd.DataFrame(rows)
if boundary_df.empty:
    raise ValueError('Boundary evaluation produced no rows.')

inside_total = int(boundary_df['inside_center_count'].sum())
outside_total = int(boundary_df['outside_center_count'].sum())
full_total = int(boundary_df['fullframe_detection_count'].sum())

summary = {
    'audit_id': 'roi_boundary_adherence_eval_v1',
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'weights_path': str(weights_path),
    'eval_config': cfg,
    'num_rois': int(len(boundary_df)),
    'num_unique_images': int(boundary_df['image_path'].nunique()),
    'inside_detections_total': inside_total,
    'outside_detections_total_before_filter': outside_total,
    'fullframe_detections_total': full_total,
    'outside_share_before_filter': float(outside_total / full_total) if full_total > 0 else 0.0,
    'rois_with_inside_detection': int((boundary_df['inside_center_count'] > 0).sum()),
    'rois_with_outside_detection_before_filter': int((boundary_df['outside_center_count'] > 0).sum()),
    'near_edge_inside_detection_rate': float(boundary_df['near_edge_inside_count'].sum() / max(1, inside_total)),
    'center_in_roi_filter_policy': {
        'rule': 'keep detections whose box center is inside ROI',
        'outside_outputs_after_filter': 0,
        'note': 'Outside-boundary detections are removed by construction with this policy.',
    },
    'by_split': {},
}

for split, part in boundary_df.groupby('split'):
    p_inside = int(part['inside_center_count'].sum())
    p_outside = int(part['outside_center_count'].sum())
    p_total = int(part['fullframe_detection_count'].sum())
    summary['by_split'][str(split)] = {
        'num_rois': int(len(part)),
        'inside_detections_total': p_inside,
        'outside_detections_total_before_filter': p_outside,
        'outside_share_before_filter': float(p_outside / p_total) if p_total > 0 else 0.0,
        'rois_with_inside_detection': int((part['inside_center_count'] > 0).sum()),
        'rois_with_outside_detection_before_filter': int((part['outside_center_count'] > 0).sum()),
    }

_write_json(RUN_DIRS['metrics'] / 'roi_boundary_adherence_summary.json', summary)
boundary_df.to_csv(RUN_DIRS['metrics'] / 'roi_boundary_adherence_details.csv', index=False)

md_lines = [
    '# ROI Boundary-Adherence Evaluation v1',
    f"- run_id: `{RUN_ID}`",
    f"- num_rois: `{summary['num_rois']}`",
    f"- unique_images: `{summary['num_unique_images']}`",
    f"- outside_share_before_filter: `{summary['outside_share_before_filter']:.4f}`",
    f"- rois_with_outside_detection_before_filter: `{summary['rois_with_outside_detection_before_filter']}`",
    f"- center-in-roi filter outside outputs: `0` (by policy)",
    '',
    '## By Split',
]
for split, s in summary['by_split'].items():
    md_lines.append(
        f"- {split}: rois={s['num_rois']}, outside_share_before_filter={s['outside_share_before_filter']:.4f}, "
        f"rois_with_outside={s['rois_with_outside_detection_before_filter']}"
    )

(RUN_DIRS['reports'] / 'roi_boundary_adherence_report.md').write_text('\\n'.join(md_lines), encoding='utf-8')

print('Using existing run:', RUN_ROOT)
print('ROI boundary summary:', RUN_DIRS['metrics'] / 'roi_boundary_adherence_summary.json')
print('ROI boundary details:', RUN_DIRS['metrics'] / 'roi_boundary_adherence_details.csv')
print('ROI boundary report:', RUN_DIRS['reports'] / 'roi_boundary_adherence_report.md')
print('ROIs evaluated:', len(boundary_df))
print('Outside-share before ROI filter:', round(summary['outside_share_before_filter'], 4))


## 15) Assistive Spoken-Burden Evaluation (Issue 4)
Quantifies narration load as detections-per-RoI using outputs from Issue 3 and background FP results from Issue 1.


In [ ]:
# Assistive spoken-burden evaluation (Issue 4, standalone)
import os
import json
from collections import Counter
from pathlib import Path

import pandas as pd


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def _pick_existing_run_dir(dataset_root: Path) -> Path:
    runs_root = dataset_root / 'experiments' / 'colab_runs'
    if not runs_root.exists():
        raise FileNotFoundError(f'Run root not found: {runs_root}')
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No existing run folders found in: {runs_root}')
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def _pct(series_bool):
    return float(series_bool.mean()) if len(series_bool) > 0 else 0.0


DATASET_ROOT = Path(os.environ.get('GALVAT_DATASET_ROOT', '/content/drive/MyDrive/galvat_openimages_v7'))
existing_run_override = os.environ.get('GALVAT_EXISTING_RUN_DIR', '').strip()
RUN_ROOT = Path(existing_run_override) if existing_run_override else _pick_existing_run_dir(DATASET_ROOT)
RUN_ID = RUN_ROOT.name
RUN_DIRS = {
    'root': RUN_ROOT,
    'logs': RUN_ROOT / 'logs',
    'metrics': RUN_ROOT / 'metrics',
    'models': RUN_ROOT / 'models',
    'exports': RUN_ROOT / 'exports',
    'env': RUN_ROOT / 'env',
    'config': RUN_ROOT / 'config',
    'reports': RUN_ROOT / 'reports',
    'snapshots': RUN_ROOT / 'snapshots',
}
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

roi_details_path = RUN_DIRS['metrics'] / 'roi_boundary_adherence_details.csv'
if not roi_details_path.exists():
    raise FileNotFoundError(
        f'Missing {roi_details_path}. Run Issue 3 cell first to produce structured ROI detections.'
    )

roi_df = pd.read_csv(roi_details_path)
if roi_df.empty:
    raise ValueError('ROI boundary details are empty.')

budget = int(os.environ.get('GALVAT_NARRATION_BUDGET_PER_ROI', '2'))
hard_budget = int(os.environ.get('GALVAT_NARRATION_HARD_BUDGET_PER_ROI', '3'))

object_counts = roi_df['inside_center_count'].fillna(0).astype(int)

class_counter = Counter()
for raw in roi_df.get('inside_classes_json', pd.Series([], dtype='object')).fillna('[]').tolist():
    try:
        cls_list = json.loads(raw)
        if isinstance(cls_list, list):
            for cls_name in cls_list:
                class_counter[str(cls_name)] += 1
    except Exception:
        continue

summary = {
    'analysis_id': 'assistive_spoken_burden_v1',
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'narration_policy': {
        'target_max_detections_per_roi': budget,
        'hard_max_detections_per_roi': hard_budget,
    },
    'object_roi_metrics': {
        'num_rois': int(len(object_counts)),
        'mean_detections_per_roi': float(object_counts.mean()),
        'median_detections_per_roi': float(object_counts.median()),
        'p90_detections_per_roi': float(object_counts.quantile(0.90)),
        'p95_detections_per_roi': float(object_counts.quantile(0.95)),
        'max_detections_per_roi': int(object_counts.max()),
        'pct_rois_with_any_detection': _pct(object_counts > 0),
        'pct_rois_over_target_budget': _pct(object_counts > budget),
        'pct_rois_over_hard_budget': _pct(object_counts > hard_budget),
    },
    'background_roi_metrics': None,
    'top_spoken_classes': [
        {'class': cls_name, 'mentions': int(cnt)} for cls_name, cnt in class_counter.most_common(20)
    ],
    'by_split': {},
}

for split, part in roi_df.groupby('split'):
    c = part['inside_center_count'].fillna(0).astype(int)
    summary['by_split'][str(split)] = {
        'num_rois': int(len(c)),
        'mean_detections_per_roi': float(c.mean()),
        'pct_rois_over_target_budget': _pct(c > budget),
        'pct_rois_over_hard_budget': _pct(c > hard_budget),
    }

bg_details_path = RUN_DIRS['metrics'] / 'background_fp_details.csv'
if bg_details_path.exists():
    bg_df = pd.read_csv(bg_details_path)
    if not bg_df.empty and 'detections' in bg_df.columns:
        bg_counts = bg_df['detections'].fillna(0).astype(int)
        summary['background_roi_metrics'] = {
            'num_rois': int(len(bg_counts)),
            'mean_detections_per_roi': float(bg_counts.mean()),
            'pct_rois_with_any_detection': _pct(bg_counts > 0),
            'high_risk_false_narration_rate': _pct(bg_counts >= 1),
        }

summary['acceptance'] = {
    'target_budget_met': summary['object_roi_metrics']['pct_rois_over_target_budget'] <= 0.10,
    'hard_budget_met': summary['object_roi_metrics']['pct_rois_over_hard_budget'] <= 0.02,
    'background_fp_risk_note': 'background metrics unavailable' if summary['background_roi_metrics'] is None else 'background metrics included',
}

_write_json(RUN_DIRS['metrics'] / 'assistive_spoken_burden_summary.json', summary)

spoken_rows = roi_df[[
    'split', 'image_id', 'inside_center_count', 'outside_center_count', 'fullframe_detection_count'
]].copy()
spoken_rows = spoken_rows.rename(columns={'inside_center_count': 'narrated_detection_count'})
spoken_rows.to_csv(RUN_DIRS['metrics'] / 'assistive_spoken_burden_per_roi.csv', index=False)

md_lines = [
    '# Assistive Spoken-Burden Evaluation v1',
    f"- run_id: `{RUN_ID}`",
    f"- target_budget_per_roi: `{budget}`",
    f"- hard_budget_per_roi: `{hard_budget}`",
    f"- mean_detections_per_roi: `{summary['object_roi_metrics']['mean_detections_per_roi']:.4f}`",
    f"- p95_detections_per_roi: `{summary['object_roi_metrics']['p95_detections_per_roi']:.4f}`",
    f"- pct_rois_over_target_budget: `{summary['object_roi_metrics']['pct_rois_over_target_budget']:.4f}`",
    f"- pct_rois_over_hard_budget: `{summary['object_roi_metrics']['pct_rois_over_hard_budget']:.4f}`",
]
if summary['background_roi_metrics'] is not None:
    md_lines.extend([
        f"- background_pct_rois_with_any_detection: `{summary['background_roi_metrics']['pct_rois_with_any_detection']:.4f}`",
        f"- background_mean_detections_per_roi: `{summary['background_roi_metrics']['mean_detections_per_roi']:.4f}`",
    ])

md_lines.extend(['', '## Top Spoken Classes'])
for row in summary['top_spoken_classes'][:15]:
    md_lines.append(f"- {row['class']}: {row['mentions']}")

(RUN_DIRS['reports'] / 'assistive_spoken_burden_report.md').write_text('\\n'.join(md_lines), encoding='utf-8')

print('Using existing run:', RUN_ROOT)
print('Spoken-burden summary:', RUN_DIRS['metrics'] / 'assistive_spoken_burden_summary.json')
print('Spoken-burden per-ROI CSV:', RUN_DIRS['metrics'] / 'assistive_spoken_burden_per_roi.csv')
print('Spoken-burden report:', RUN_DIRS['reports'] / 'assistive_spoken_burden_report.md')
print('Mean detections/ROI:', round(summary['object_roi_metrics']['mean_detections_per_roi'], 4))
print('Pct over target budget:', round(summary['object_roi_metrics']['pct_rois_over_target_budget'], 4))


## 16) Coworker Deployment Package Check (Issue 6)
Builds a complete handoff bundle manifest (ONNX path, label map, confidence/NMS thresholds, and inference config) for coworker Unity integration.


In [ ]:
# Coworker deployment package check (Issue 6, standalone)
import os
import json
import hashlib
from pathlib import Path

import yaml


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def _safe_read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def _pick_existing_run_dir(dataset_root: Path) -> Path:
    runs_root = dataset_root / 'experiments' / 'colab_runs'
    if not runs_root.exists():
        raise FileNotFoundError(f'Run root not found: {runs_root}')
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No existing run folders found in: {runs_root}')
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def _sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _file_row(path: Path):
    return {
        'path': str(path),
        'exists': bool(path.exists()),
        'size_bytes': int(path.stat().st_size) if path.exists() else None,
        'sha256': _sha256_file(path) if path.exists() else None,
    }


DATASET_ROOT = Path(os.environ.get('GALVAT_DATASET_ROOT', '/content/drive/MyDrive/galvat_openimages_v7'))
existing_run_override = os.environ.get('GALVAT_EXISTING_RUN_DIR', '').strip()
RUN_ROOT = Path(existing_run_override) if existing_run_override else _pick_existing_run_dir(DATASET_ROOT)
RUN_ID = RUN_ROOT.name
RUN_DIRS = {
    'root': RUN_ROOT,
    'logs': RUN_ROOT / 'logs',
    'metrics': RUN_ROOT / 'metrics',
    'models': RUN_ROOT / 'models',
    'exports': RUN_ROOT / 'exports',
    'env': RUN_ROOT / 'env',
    'config': RUN_ROOT / 'config',
    'reports': RUN_ROOT / 'reports',
    'snapshots': RUN_ROOT / 'snapshots',
}
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

train_summary_path = RUN_DIRS['metrics'] / 'train_summary.json'
best_pt = None
if train_summary_path.exists():
    try:
        best_pt_candidate = Path(_safe_read_json(train_summary_path).get('best_weights', ''))
        if best_pt_candidate.exists():
            best_pt = best_pt_candidate
    except Exception:
        pass
if best_pt is None:
    best_candidates = sorted(RUN_ROOT.rglob('best.pt'), key=lambda x: x.stat().st_mtime, reverse=True)
    best_pt = best_candidates[0] if best_candidates else Path('')

onnx_path = None
onnx_summary_path = RUN_DIRS['exports'] / 'onnx_export_summary.json'
if onnx_summary_path.exists():
    try:
        onnx_candidate = Path(_safe_read_json(onnx_summary_path).get('onnx_path', ''))
        if onnx_candidate.exists():
            onnx_path = onnx_candidate
    except Exception:
        pass
if onnx_path is None:
    onnx_candidates = sorted(RUN_ROOT.rglob('*.onnx'), key=lambda x: x.stat().st_mtime, reverse=True)
    onnx_path = onnx_candidates[0] if onnx_candidates else Path('')

classes_txt = DATASET_ROOT / 'config' / 'classes.txt'
class_to_idx_json = DATASET_ROOT / 'config' / 'class_to_idx.json'
data_yaml_path = DATASET_ROOT / 'config' / 'data.yaml'

if classes_txt.exists():
    class_names = [x.strip() for x in classes_txt.read_text(encoding='utf-8').splitlines() if x.strip()]
elif data_yaml_path.exists():
    raw = yaml.safe_load(data_yaml_path.read_text(encoding='utf-8'))
    names = raw.get('names', [])
    class_names = [str(x) for x in names]
else:
    class_names = []

if class_to_idx_json.exists():
    class_to_idx = _safe_read_json(class_to_idx_json)
else:
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

run_cfg_path = RUN_DIRS['config'] / 'run_config.json'
train_args_path = RUN_DIRS['config'] / 'train_args_effective.json'

run_cfg = _safe_read_json(run_cfg_path) if run_cfg_path.exists() else {}
train_cfg = run_cfg.get('train_config', {}) if isinstance(run_cfg, dict) else {}
train_args = _safe_read_json(train_args_path) if train_args_path.exists() else {}

confidence_threshold = float(os.environ.get('GALVAT_DEPLOY_CONF', train_cfg.get('conf', 0.25)))
nms_iou_threshold = float(os.environ.get('GALVAT_DEPLOY_NMS_IOU', train_cfg.get('iou', 0.45)))
imgsz = int(train_args.get('imgsz', train_cfg.get('imgsz', 640)))

bundle_dir = RUN_DIRS['exports'] / 'coworker_handoff_bundle_v1'
bundle_dir.mkdir(parents=True, exist_ok=True)

label_map_payload = {
    'run_id': RUN_ID,
    'class_count': len(class_names),
    'class_names': class_names,
    'class_to_idx': class_to_idx,
}
_write_json(bundle_dir / 'label_map.json', label_map_payload)

inference_cfg = {
    'run_id': RUN_ID,
    'device_target': 'Magic Leap 2 Enterprise (OpenXR baseline)',
    'model': {
        'onnx_path': str(onnx_path),
        'pytorch_best_path': str(best_pt),
        'input_resolution': [imgsz, imgsz],
    },
    'postprocess': {
        'confidence_threshold': confidence_threshold,
        'nms_iou_threshold': nms_iou_threshold,
        'max_detections': int(train_cfg.get('max_det', 100)),
    },
    'labels': {
        'class_count': len(class_names),
        'label_map_path': str(bundle_dir / 'label_map.json'),
    },
}
_write_json(bundle_dir / 'inference_config.json', inference_cfg)

manifest_files = [
    best_pt,
    onnx_path,
    classes_txt,
    class_to_idx_json,
    data_yaml_path,
    bundle_dir / 'label_map.json',
    bundle_dir / 'inference_config.json',
    run_cfg_path,
    train_args_path,
]
manifest_payload = {
    'bundle_id': 'coworker_deployment_bundle_v1',
    'run_id': RUN_ID,
    'bundle_root': str(bundle_dir),
    'files': [_file_row(Path(p)) for p in manifest_files],
}
_write_json(bundle_dir / 'bundle_manifest.json', manifest_payload)

missing_checks = []
if not onnx_path.exists():
    missing_checks.append('onnx_model')
if len(class_names) == 0:
    missing_checks.append('label_map')
if confidence_threshold <= 0:
    missing_checks.append('confidence_threshold')
if nms_iou_threshold <= 0:
    missing_checks.append('nms_iou_threshold')

package_check = {
    'check_id': 'coworker_deployment_package_check_v1',
    'run_id': RUN_ID,
    'status': 'pass' if not missing_checks else 'fail',
    'missing_items': missing_checks,
    'key_outputs': {
        'onnx_path': str(onnx_path),
        'bundle_manifest': str(bundle_dir / 'bundle_manifest.json'),
        'label_map': str(bundle_dir / 'label_map.json'),
        'inference_config': str(bundle_dir / 'inference_config.json'),
    },
    'thresholds': {
        'confidence_threshold': confidence_threshold,
        'nms_iou_threshold': nms_iou_threshold,
    },
}
_write_json(RUN_DIRS['metrics'] / 'coworker_deployment_package_check_v1.json', package_check)

readme_lines = [
    '# GALVAT Coworker Deployment Bundle v1',
    f'- run_id: `{RUN_ID}`',
    f'- onnx_path: `{onnx_path}`',
    f'- label_map: `{bundle_dir / "label_map.json"}`',
    f'- inference_config: `{bundle_dir / "inference_config.json"}`',
    f'- bundle_manifest: `{bundle_dir / "bundle_manifest.json"}`',
    '',
    '## Unity/Magic Leap Handoff Notes',
    '- Use `inference_config.json` thresholds as baseline runtime defaults.',
    '- Keep class index order exactly as in `label_map.json`.',
    '- If runtime tuning changes confidence/NMS thresholds, record the delta in a new handoff note file.',
]
(bundle_dir / 'README.md').write_text('\\n'.join(readme_lines), encoding='utf-8')

print('Using existing run:', RUN_ROOT)
print('Deployment package check:', RUN_DIRS['metrics'] / 'coworker_deployment_package_check_v1.json')
print('Bundle manifest:', bundle_dir / 'bundle_manifest.json')
print('Label map:', bundle_dir / 'label_map.json')
print('Inference config:', bundle_dir / 'inference_config.json')
print('Status:', package_check['status'])
if missing_checks:
    print('Missing:', ', '.join(missing_checks))


## 17) Optional: Mirror Artifacts Into Repo for Agents
If `REPO_ROOT` exists, this copies the full run bundle into `domains/galvat/artifacts/files/<run_id>/`.


In [ ]:
if REPO_ROOT is not None and REPO_ROOT.exists():
    target = REPO_ROOT / 'domains' / 'galvat' / 'artifacts' / 'files' / RUN_ID
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(RUN_ROOT, target)

    pointer = {
        'run_id': RUN_ID,
        'colab_run_root': str(RUN_ROOT),
        'repo_artifact_root': str(target),
        'timestamp_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }
    pointer_path = REPO_ROOT / 'domains' / 'galvat' / 'artifacts' / 'files' / f'{RUN_ID}_pointer.json'
    pointer_path.write_text(json.dumps(pointer, indent=2), encoding='utf-8')

    print('Mirrored run bundle to repo path:', target)
    print('Pointer file:', pointer_path)
else:
    print('REPO_ROOT not found; skipping mirror step.')


## 18) Final Output
At this point, all major outputs are written under:
- `galvat_openimages_v7/experiments/colab_runs/<run_id>/`
This includes:
- run config
- dataset snapshot
- environment info
- train/val/test metrics
- per-class metrics
- ONNX export summary
- file checksum manifest
- agent analysis bundle
- background false-positive summary + detailed rows
- background ROI slice build summary
- ROI boundary-adherence summary + detailed rows
- assistive spoken-burden summary + per-RoI rows
- coworker deployment bundle check + handoff manifest
